In [1]:
pip install rdflib owlready2 -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install  graphviz pydotplus -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
!apt-get install -y openjdk-11-jdk   -qq

"apt-get" не является внутренней или внешней
командой, исполняемой программой или пакетным файлом.


In [4]:
import pandas as pd
import os
from owlready2 import *

## Загрузка данных
DataProperty (свойство-данное) — связывает объект с конкретным значением (числом, строкой, датой).
Пример: имя_сотрудника — у каждого специалиста есть имя (строка).
Аналог в обычной базе данных: колонка full_name в таблице Employees.

ObjectProperty (объектное свойство) — связывает один объект с другим объектом (индивидом).
Пример: сотрудник → обладает_знанием → знание. Это не число и не строка, а ссылка на другой объект (знание «Python»).
Аналог в реляционной БД: внешний ключ knowledge_id в таблице EmployeeKnowledge, но без необходимости писать JOIN — вы просто идёте по связи.

Как читать свойства (по коду):

domain — кто может иметь это свойство (от кого исходит стрелка).

range — на кого или на что указывает свойство (тип значения)

In [5]:
import os

def load_data():
    base_path = r"C:\Users\User\Desktop\учеба\6 семестр\Большие данные в цифровой экономике\big_data\task_8_Ontology\ontology_data\ontology_data"
    knowledge_df = pd.read_csv(os.path.join(base_path, 'knowledge.csv'))
    competencies_df = pd.read_csv(os.path.join(base_path, 'competencies.csv'))
    competency_knowledge_df = pd.read_csv(os.path.join(base_path, 'competency_knowledge.csv'))
    positions_df = pd.read_csv(os.path.join(base_path, 'positions.csv'))
    position_competency_df = pd.read_csv(os.path.join(base_path, 'position_competency.csv'))
    employees_df = pd.read_csv(os.path.join(base_path, 'employees.csv'))
    return knowledge_df, competencies_df, competency_knowledge_df, positions_df, position_competency_df, employees_df

In [6]:
knowledge_df, competencies_df, competency_knowledge_df, positions_df, position_competency_df, employees_df = load_data()

In [7]:
knowledge_df

,knowledge_id,knowledge_name
0,1,Математическая статистика
1,2,Линейная алгебра
2,3,Теория вероятностей
3,4,Python программирование
4,5,"Библиотеки ML (scikit-learn, TensorFlow, PyTorch)"
5,6,Обработка естественного языка (NLP)
6,7,Компьютерное зрение
7,8,"Работа с большими данными (Spark, Hadoop)"
8,9,SQL и базы данных
9,10,Визуализация данных


# Создание новой онтологии

## Итерация 1.
Создадим знания и компетенции

In [8]:
# Создаем онтологию - создаёт новую онтологию с этим идентификатором.
onto = get_ontology("http://test.org/ml_specialist_v1.owl")

#контекстный менеджер. Внутри этого блока все создаваемые классы, свойства и индивиды автоматически привязываются к онтологии onto
with onto:
    #объявляет класс (понятие) с именем Знание. Thing — это встроенный в owlready2 корневой класс, от которого наследуются все классы в онтологии.
    class Знание(Thing):
        pass #тело класса пустое, так как все свойства будут добавлены отдельно. 

    class Компетенция(Thing): #Аналогично создаётся класс Компетенция
        pass

#Смысл: каждая компетенция «требует знание» (или несколько знаний)
    
    #ObjectProperty – означает, что свойство связывает один объект (индивид) с другим объектом.
    class требует_знание(ObjectProperty): #создаёт объектное свойство (связь) между двумя классами.
        domain = [Компетенция] #область определения (domain) этого свойства — класс Компетенция. 
                                #То есть свойство может быть применено только к индивидам класса Компетенция
        range = [Знание] #указывает, что область значений (range) — класс Знание. То есть свойство связывает компетенцию с каким-то знанием.

#Смысл: у любого объекта (знания, компетенции, сотрудника) будет удобочитаемое название, записанное строкой.

    #DataProperty – в отличие от ObjectProperty, ведёт к литералу (строка, число, дата), а не к другому объекту.
    class название(DataProperty):#создаёт дата-свойство (атрибут), которое связывает объект с конкретным значением (строкой, числом и т.д.).
        domain = [Thing] #область определения — любой класс (т.к. Thing — корень всех классов).
                        #Значит, любая сущность (знание, компетенция, специалист, должность) может иметь это свойство.
        range = [str] #значение свойства должно быть строкой (str).

### Создание индивидуальностей для знаний и компетенций



In [9]:
#превращают строки из CSV‑таблиц в объекты онтологии (индивиды). Каждая строка таблицы становится отдельным экземпляром класса Знание или Компетенция.
def create_knowledge(knowledge_df): #функция принимает DataFrame, который содержит колонки knowledge_id и knowledge_name
  with onto: #снова открываем контекст онтологии, чтобы создаваемые объекты автоматически принадлежали онтологии onto

    #пустой словарь, где ключом будет ID знания, а значением – созданный объект онтологии.
    knowledge_instances = {}
    #цикл по каждой строке DataFrame. _ означает, что нам не нужен индекс строки, row – это строка.
    for _, row in knowledge_df.iterrows():
        knowledge = Знание(f"знание_{row['knowledge_id']}") #создаём индивид (конкретный экземпляр) класса Знание. Имя индивида формируется как "знание_1", "знание_2" и т.д. (например, знание_1). имя индивида должно быть уникальным в пределах онтологии.
        knowledge.название = [row['knowledge_name']] #заполняем дата-свойство название для этого индивида. Важно: свойство может иметь несколько значений, поэтому справа ожидается список. Здесь мы передаём один элемент – название знания из CSV
        knowledge_instances[row['knowledge_id']] = knowledge #охраняем созданный объект в словарь под ключом knowledge_id, чтобы потом легко обращаться к знанию по его ID.
    return knowledge_instances

def create_competencies( competencies_df):
  with onto:
    competency_instances = {}
    for _, row in competencies_df.iterrows():
        competency = Компетенция(f"компетенция_{row['competency_id']}")
        competency.название = [row['competency_name']]
        competency_instances[row['competency_id']] = competency
    return competency_instances
#нужно, чтобы в последующем связать компетенции со знаниями

In [10]:
knowledges = create_knowledge(knowledge_df)

In [11]:
knowledges

#1, 2, 3, … 10 – это knowledge_id из CSV-файла.
#Значения словаря – это объекты-индивиды (экземпляры) класса Знание, созданные в онтологии.
#ml_specialist_v1.знание_1 – полное имя индивида: онтология ml_specialist_v1, индивид с именем знание_1.

{1: ml_specialist_v1.знание_1,
 2: ml_specialist_v1.знание_2,
 3: ml_specialist_v1.знание_3,
 4: ml_specialist_v1.знание_4,
 5: ml_specialist_v1.знание_5,
 6: ml_specialist_v1.знание_6,
 7: ml_specialist_v1.знание_7,
 8: ml_specialist_v1.знание_8,
 9: ml_specialist_v1.знание_9,
 10: ml_specialist_v1.знание_10}

In [12]:
#выводит на экран названия всех знаний, которые были загружены из CSV и сохранены в словарь knowledges.
#.items() возвращает пары (ключ, значение) для каждой записи в словаре.
for id, item in knowledges.items():
  print (item.название[0]) #из-за того, что свойство может хранить несколько значений, 
                        #библиотека всегда упаковывает значение в список. Чтобы получить «чистую» строку, нужно взять нулевой элемент списка, иначе 
                        #будет в квадртаных скобках 

Математическая статистика
Линейная алгебра
Теория вероятностей
Python программирование
Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Обработка естественного языка (NLP)
Компьютерное зрение
Работа с большими данными (Spark, Hadoop)
SQL и базы данных
Визуализация данных


In [13]:
#то же самое, но для компетенций
competencies = create_competencies( competencies_df) #создаёт словарь, где ключ – competency_id (число), 
                                                    #значение – объект-компетенция (например, ml_specialist_v1.компетенция_1).
for id, item in competencies.items():
  print (item.название[0]) #из объекта компетенции берёт её название (первый элемент списка, так как owlready2 хранит значения в списке).

Математическая подготовка
Программирование и алгоритмы
Машинное обучение
Глубокое обучение
Обработка и анализ данных
Работа с большими данными
Визуализация и представление данных
Развертывание ML-моделей


### Создание отношений между знаниями и компетенциями

In [14]:
# Связываем компетенции со знаниями
#competency_knowledge_df – DataFrame из CSV, где в каждой строке указаны competency_id и knowledge_id. Например, строка (1, 1) означает, что компетенция №1 
#(«Математическая подготовка») требует знание №1 («Математическая статистика»).
#competency_instances – словарь, который вернула функция create_competencies()
#knowledge_instances – словарь, который вернула функция create_knowledge()
def  relate_competencies_knowledge (competency_knowledge_df,  competency_instances, knowledge_instances):
  with onto:
      #Цикл по каждой строке таблицы. _ означает, что индекс строки нам не нужен, row – это сама строка с колонками competency_id и knowledge_id.
    for _, row in competency_knowledge_df.iterrows():
        #По ID из строки находим объект компетенции и объект знания в соответствующих словарях.
        competency = competency_instances[row['competency_id']]
        knowledge = knowledge_instances[row['knowledge_id']]
        #Это объектное свойство, которое связывает компетенцию со знанием. Метод .append() добавляет к этому свойству новое значение – объект знания.
        competency.требует_знание.append(knowledge)


In [15]:
#запускает функцию, которая для каждой строки из файла competency_knowledge.csv связывает компетенцию с нужным знанием.
relate_competencies_knowledge (competency_knowledge_df, competencies, knowledges)

Выведем связи между компенциями и знаниями

In [16]:
def print_competency_knowledge():
  print("\n" + "="*60)
  print("СВЯЗИ ЗНАНИЙ И КОМПЕТЕНЦИЙ")
  print("="*60)

    #Перебирает словарь competencies, где ключ – ID компетенции (число), а значение – объект компетенции в онтологии 
  for competency_id, competency in competencies.items():
      #Из объекта компетенции берёт её название.
      comp_name = competency.название[0]
      print(f"\n{comp_name} требует знания:")

      #competency.требует_знание – это объектное свойство, которое хранит список знаний (объектов класса Знание), связанных с данной компетенцией.
      required_knowledge = list(competency.требует_знание)
      for know in required_knowledge:
          print(f"  - {know.название[0]}")

print_competency_knowledge()



СВЯЗИ ЗНАНИЙ И КОМПЕТЕНЦИЙ

Математическая подготовка требует знания:
  - Математическая статистика
  - Линейная алгебра
  - Теория вероятностей

Программирование и алгоритмы требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Машинное обучение требует знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Глубокое обучение требует знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Обработка и анализ данных требует знания:
  - Математическая статистика
  - SQL и базы данных

Работа с большими данными требует знания:
  - Работа с большими данными (Spark, Hadoop)
  - SQL и базы данных

Визуализация и представление данных требует знания:
  - Визуализация данных
  - Python программирование

Развертывание ML-моделей требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, 

# Итерация 2

Добавление новых концептов в онтологию


In [17]:
with onto:
    #классы
  class Специалист(Thing):
        pass

  class Должность(Thing):
        pass

  class обладает_знанием(ObjectProperty):
      #domain – свойство может применяться только к Специалист.
        domain = [Специалист]
      #range – значение свойства должно быть объектом класса Знание.
        range = [Знание]

    #Связывает сотрудника с его компетенциями (которые выводятся на основе знаний).
  class обладает_компетенцией(ObjectProperty):
        domain = [Специалист]
        range = [Компетенция]

    #Показывает, какие должности подходят сотруднику на основе его компетенций.
  class может_занимать_должность(ObjectProperty):
        domain = [Специалист]
        range = [Должность]

    #Задаёт требования к компетенциям для каждой должности 
    #(например, для должности ML Engineer нужны компетенции «Программирование и алгоритмы» и «Машинное обучение»).
  class требует_компетенцию(ObjectProperty):
        domain = [Должность]
        range = [Компетенция]

  class имя_сотрудника(DataProperty):
        domain = [Специалист]
        range = [str]

In [18]:
#группирует записи о сотрудниках по идентификатору и имени, а затем собирает все knowledge_id для каждого сотрудника в один список.

#Это означает, что у Иванова есть три знания (ID 1, 4, 5). Чтобы в онтологии создать одного сотрудника и привязать к нему все его знания,
#нужно сначала сгруппировать строки и превратить knowledge_id в список.
employees_df.groupby(['employee_id', 'full_name'])['knowledge_id'].apply(list)

employee_id  full_name                     
1            Иванов Алексей Сергеевич           [1, 4, 5]
2            Петрова Мария Владимировна         [1, 4, 5]
3            Сидоров Дмитрий Петрович           [1, 2, 9]
4            Козлова Анна Игоревна              [5, 6, 7]
5            Федоров Максим Андреевич           [5, 6, 7]
6            Николаева Екатерина Дмитриевна    [4, 5, 10]
7            Орлов Сергей Викторович            [1, 2, 3]
Name: knowledge_id, dtype: object

### Создание индивидуальностей для сотрудников и их отношений с знаниями


In [19]:
#функция превращает данные о сотрудниках из CSV в объекты-специалистов внутри онтологии. Каждому сотруднику она присваивает имя и все его знания.
def create_employees(employees_df, knowledge_instances):
    #Удаляет возможные дубликаты строк в исходной таблице 
  employees_df = employees_df.drop_duplicates()
#Группирует строки по сотруднику (уникальные пары employee_id, full_name
  employee_knowledge = employees_df.groupby(['employee_id', 'full_name'])['knowledge_id'] \
    .apply(lambda x: list(set(x))).reset_index()
    #Для каждого сотрудника собирает уникальные knowledge_id в список
    #apply(...) – применяет функцию к каждой группе.
    #lambda x: ... – безымянная функция, где x – это все значения knowledge_id в текущей группе (например, [1,4,5,1]).
    #set(x) – превращает список x в множество. Множество автоматически удаляет дубликаты.
    #list(...) – превращает множество обратно в список.

    #Создаёт пустой словарь, в который мы сложим всех созданных сотрудников.
  employee_instances = {}
    #Проходит по каждой строке таблицы (по каждому сотруднику).
  for _, row in employee_knowledge.iterrows():
    # Создаёт индивида класса Специалист с уникальным именем
      employee = Специалист(f"сотрудник_{row['employee_id']}")
      #Записывает дата-свойство имя_сотрудника (список с одним элементом — строкой с ФИО).
      employee.имя_сотрудника = [row['full_name']]
      #Сохраняет созданного сотрудника в словарь
      employee_instances[row['employee_id']] = employee

      # Перебирает все ID знаний, которые есть у этого сотрудника.
      for knowledge_id in row['knowledge_id']:
          #Если такой ID есть в словаре knowledge_instances (а там лежат все объекты Знание), то добавляет связь обладает_знанием — прикрепляет объект знания к сотруднику.
          if knowledge_id in knowledge_instances:
              employee.обладает_знанием.append(knowledge_instances[knowledge_id])

  return employee_instances

In [20]:
#Переменная employees становится словарём, например:
#{1: ml_specialist_v1.сотрудник_1, 2: ml_specialist_v1.сотрудник_2, ...}
employees = create_employees(employees_df, knowledges)

In [21]:
def print_employee_mindmap():
  print("\n" + "="*60)
  print("КАРТА ЗНАНИЙ КАЖДОГО СОТРУДНИКА")
  print("="*60)
    #Перебирает словарь employees, который вернула функция create_employees().
  for employee_id, employee in employees.items():
      #employee.имя_сотрудника – список (так как owlready2 хранит значения в списке).
    #[0] – первый элемент списка (строка с ФИО).
      #Тернарный оператор if ... else ... на случай, если свойства нет – тогда подставляется "Сотрудник X".
          employee_name = employee.имя_сотрудника[0] if employee.имя_сотрудника else f"Сотрудник {employee_id}"

          print(f"\n{employee_name}:")

          # Получаем список знаний сотрудника
          knowledge_list = list(employee.обладает_знанием)

      #Если список знаний не пуст
          if knowledge_list:
              print("Знания:")
              for knowledge in knowledge_list:
                  #Для каждого объекта знания берёт его название: knowledge.название[0] (снова из списка).
                  #Если по какой-то причине у знания нет названия, Тогда вместо пустой строки или ошибки вы увидите хотя бы техническое имя (знание_1
                  know_name = knowledge.название[0] if knowledge.название else knowledge.name
                  print(f"  - {know_name}")
          else:
              print("Знания: нет")

print_employee_mindmap()


КАРТА ЗНАНИЙ КАЖДОГО СОТРУДНИКА

Иванов Алексей Сергеевич:
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Петрова Мария Владимировна:
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Сидоров Дмитрий Петрович:
Знания:
  - Математическая статистика
  - Линейная алгебра
  - SQL и базы данных

Козлова Анна Игоревна:
Знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Федоров Максим Андреевич:
Знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Николаева Екатерина Дмитриевна:
Знания:
  - Визуализация данных
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Орлов Сергей Викторович:
Знания:
  - Математическая статистика
  - Линейная алгебра
  - Теория вероятностей


### Создание индивидуальностей для должностей и из связей с компетенциями


In [22]:
# Создаем экземпляры должностей
#position_instances = {} – создаётся пустой словарь, который будет хранить созданные объекты должностей (ключ – position_id, значение – объект Должность).
def create_position_instances(positions_df):
    position_instances = {}
    #цикл по каждой строке 
    for _, row in positions_df.iterrows():
        #создаётся индивид (конкретный экземпляр) класса Должность с именем вида "должность_1", "должность_2" и т.д.
        position = Должность(f"должность_{row['position_id']}")
        #присваивается атрибут name (не путать со свойством название – здесь используется просто поле объекта Python, не свойство онтологии)
        #row['position_name'] – это человекочитаемое название должности
        position.name = row['position_name']
        #сохраняется созданный объект в словарь под ключом position_id.
        position_instances[row['position_id']] = position
    return position_instances

In [23]:
positions = create_position_instances(positions_df)
positions

{1: ml_specialist_v1.ML Engineer,
 2: ml_specialist_v1.Data Scientist,
 3: ml_specialist_v1.Data Analyst,
 4: ml_specialist_v1.NLP Engineer,
 5: ml_specialist_v1.Computer Vision Engineer}

In [24]:
# Устанавливаем связи между должностями и компетенциями
def set_position_competency_relations(position_competency_df, position_instances, competency_instances):
    #Перебирает каждую строку DataFrame position_competency_df
    for _, row in position_competency_df.iterrows():
        # По ID должности находит объект Должность в словаре position_instances (который создала функция create_position_instances).
        position = position_instances[row['position_id']]
        # По ID компетенции находит объект Компетенция в словаре competency_instances (создано функцией create_competencies).
        competency = competency_instances[row['competency_id']]
        #Добавляет эту компетенцию в список требуемых для данной должности.
        #append() используется, потому что одна должность может требовать несколько компетенций.
        position.требует_компетенцию.append(competency)

In [25]:
#создаем
set_position_competency_relations(position_competency_df, positions, competencies)

In [26]:
# сюда вставить код для выполнения самостоятельных заданий
#ЗАДАНИЕ 1 
# Должность NLP Engineer (id=4) получает компетенцию "Глубокое обучение" (competencies[4])
positions[4].требует_компетенцию.append(competencies[4])

#ЗАДАНИЕ 2
# Должность Data Scientist (id=2) требует компетенцию "Математическая подготовка" (competencies[1])
positions[2].требует_компетенцию.append(competencies[1])

#ЗАДАНИЕ 3
# Сотрудник "Федоров Максим Андреевич" имеет employee_id = 5
# Для должности Computer Vision Engineer нужна компетенция "Машинное обучение" (competency_id=3)
# Для получения этой компетенции ему не хватает знаний:
#   - knowledge_id = 1 (Математическая статистика)
#   - knowledge_id = 4 (Python программирование)
# Добавляем эти знания
employees[5].обладает_знанием.append(knowledges[1])   # Математическая статистика
employees[5].обладает_знанием.append(knowledges[4])   # Python программирование

#ЗАДАНИЕ 4
# Создаём нового сотрудника
new_employee = Специалист("сотрудник_8")
new_employee.имя_сотрудника = ["Миронова Ксения Андреевна"]                

# Добавляем все существующие знания (10 штук), чтобы сотрудник мог получить все компетенции
for knowledge in knowledges.values():
    new_employee.обладает_знанием.append(knowledge)

# Добавляем нового сотрудника в словарь employees, чтобы его видели проверочные функции
employees[8] = new_employee



## Выведем требования к должностям

In [27]:
# Проверим требования должностей
def print_position_require():
  print("\n" + "="*60)
  print("ТРЕБОВАНИЯ ДОЛЖНОСТЕЙ")
  print("="*60)

    #Перебирает словарь positions, который вернула функция create_position_instances().
  for position_id, position in positions.items():
      #заполнили при создании должности (position.name = row['position_name']).
      pos_name = position.name
      print(f"\n{pos_name} требует компетенции:")

      required_competencies = list(position.требует_компетенцию)
      #Если список не пуст, для каждой компетенции из списка берётся её название через comp.название[0] (свойство онтологии).
      if required_competencies:
          for comp in required_competencies:
              print(f"  - {comp.название[0]}")
      else:
          print("  - нет требований")

print_position_require()


ТРЕБОВАНИЯ ДОЛЖНОСТЕЙ

ML Engineer требует компетенции:
  - Программирование и алгоритмы
  - Машинное обучение

Data Scientist требует компетенции:
  - Программирование и алгоритмы
  - Машинное обучение
  - Математическая подготовка

Data Analyst требует компетенции:
  - Обработка и анализ данных

NLP Engineer требует компетенции:
  - Программирование и алгоритмы
  - Глубокое обучение

Computer Vision Engineer требует компетенции:
  - Машинное обучение


## Создание аксиом для онтологии

 <b>Аксиома 1:</b> Если специалист обладает всеми требуемыми знаниями для компетенции, то у него есть эта компетенция

 <b>Аксиома 2:</b> Если у специалиста есть требуемые компетенции для должности, то он может занимать эту должность

  <b> Аксиома 3:</b> (для проверки корректности онтологии) Если специалист занимает должность, то у него должны быть соответствующие компетенции
      


In [28]:
#Эта функция создаёт три логических правила (аксиомы), которые онтология будет применять к каждому специалисту. Эти правила позволяют автоматически выводить новые знания на основе уже имеющихся.
def define_rules():
    # Аксиома 1:
    #Проверяет, обладает ли специалист всеми необходимыми знаниями для какой-либо компетенции. Если да – назначает эту компетенцию специалисту.
    with onto:
        # Объявляем класс правила с именем ИмеетКомпетенциюПоЗнаниям.
        #Специалист >> bool – это специальный синтаксис owlready2 для правил. Он означает: «Это правило применяется к объекту класса Специалист и возвращает булево значение (True/False)».
        class ИмеетКомпетенциюПоЗнаниям(Специалист >> bool):
            #Эта строчка нужна, чтобы родительский класс (Специалист >> bool) правила мог корректно инициализироваться
            #super() – обращается к родительскому классу
            #.__init__() – вызывает конструктор этого родительского класса.
            def __init__(self):
                super().__init__()
            #агический метод __call__ позволяет объекту класса вести себя как функция. Когда вы пишете rule1(специалист), на самом деле вызывается rule1.__call__(специалист).
            def __call__(self, specialist):
                # Для каждой компетенции проверяем, есть ли у специалиста все необходимые знания
                #onto.Компетенция – класс Компетенция в онтологии.
                #instances() – метод owlready2, перебираем все существующие компетенции
                for competency in onto.Компетенция.instances():
                    #competency.требует_знание – объектное свойство, которое связывает компетенцию с необходимыми знаниями. 
                    required_knowledge = list(competency.требует_знание)
                    #specialist.обладает_знанием – объектное свойство, хранящее знания сотрудника.
                    specialist_knowledge = list(specialist.обладает_знанием)

                    # Проверяем, что все требуемые знания есть у специалиста

                    #all(...) – встроенная функция Python, возвращает True, если все элементы внутри скобок истинны.
                    #knowledge in specialist_knowledge for knowledge in required_knowledge – генератор, который для каждого knowledge из списка required_knowledge проверяет, содержится ли этот объект в списке specialist_knowledge.
                    if all(knowledge in specialist_knowledge for knowledge in required_knowledge):
                        #Проверяем, нет ли уже этой компетенции у специалиста (чтобы не добавлять дубликаты).
                        #Если нет – добавляем компетенцию в список обладает_компетенцией сотрудника.
                        if competency not in specialist.обладает_компетенцией:
                            specialist.обладает_компетенцией.append(competency)

                return True

        # Аксиома 2:
        #Проверяет, есть ли у специалиста все требуемые компетенции для какой-либо должности. Если да – назначает эту должность как возможную для специалиста.
        #то же самое, только для должности
        class МожетЗаниматьДолжностьПоКомпетенциям(Специалист >> bool):
            def __init__(self):
                super().__init__()

            def __call__(self, specialist):
                #Если у специалиста есть все требуемые компетенции – добавляем должность в его список
                # Для каждой должности проверяем, есть ли у специалиста все необходимые компетенции
                for position in onto.Должность.instances():
                    required_competencies = list(position.требует_компетенцию)
                    specialist_competencies = list(specialist.обладает_компетенцией)

                    # Проверяем, что все требуемые компетенции есть у специалиста
                    if all(competency in specialist_competencies for competency in required_competencies):
                        if position not in specialist.может_занимать_должность:
                            specialist.может_занимать_должность.append(position)

                return True

        # Аксиома 3:
        #Проверяет корректность – если специалисту уже приписана какая-то должность (например, вручную), убеждается, что у него действительно есть все необходимые компетенции. Если нет – выводит предупреждение.
        class ПроверитьКомпетенцииДляДолжности(Специалист >> bool):
            def __init__(self):
                super().__init__()

            def __call__(self, specialist):
                positions = list(specialist.может_занимать_должность) #получаем список должностей, которые уже есть у специалиста
                competencies = list(specialist.обладает_компетенцией)#список его компетенций
                #Для каждой должности смотрим, какие компетенции требуются.
                for position in positions:
                    required_competencies = list(position.требует_компетенцию)
                    #Если хотя бы одна требуемая компетенция отсутствует у специалиста (not all(...)), выводим предупреждение в консоль.
                    if not all(comp in competencies for comp in required_competencies):
                        print(f"Предупреждение: у {specialist.name} нет всех компетенций для {position.name}")

                return True

    return ИмеетКомпетенциюПоЗнаниям(), МожетЗаниматьДолжностьПоКомпетенциям(), ПроверитьКомпетенцииДляДолжности()

In [29]:
rule1, rule2, rule3 = define_rules()

In [30]:
# Проверим связи знаний и компетенций
print_competency_knowledge()


СВЯЗИ ЗНАНИЙ И КОМПЕТЕНЦИЙ

Математическая подготовка требует знания:
  - Математическая статистика
  - Линейная алгебра
  - Теория вероятностей

Программирование и алгоритмы требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Машинное обучение требует знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Глубокое обучение требует знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Обработка и анализ данных требует знания:
  - Математическая статистика
  - SQL и базы данных

Работа с большими данными требует знания:
  - Работа с большими данными (Spark, Hadoop)
  - SQL и базы данных

Визуализация и представление данных требует знания:
  - Визуализация данных
  - Python программирование

Развертывание ML-моделей требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, 

In [31]:
#возвращает список всех экземпляров (индивидов) класса Специалист, которые существуют в вашей онтологии.
onto.Специалист.instances()

[ml_specialist_v1.сотрудник_1,
 ml_specialist_v1.сотрудник_2,
 ml_specialist_v1.сотрудник_3,
 ml_specialist_v1.сотрудник_4,
 ml_specialist_v1.сотрудник_5,
 ml_specialist_v1.сотрудник_6,
 ml_specialist_v1.сотрудник_7,
 ml_specialist_v1.сотрудник_8]

In [32]:
#применяет три логических правила к каждому сотруднику в онтологии, чтобы автоматически вывести его компетенции и подходящие должности, а также проверить корректность.
for employee in onto.Специалист.instances():
        rule1(employee)  # Применяем правило для компетенций
        rule2(employee)  # Применяем правило для должностей
        rule3(employee)  # Проверяем соответствие

## Проведение анализа данных

In [33]:
def print_position_for_employee():
  print("\n" + "="*60)
  print("АНАЛИЗ ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ У СОТРУДНИКОВ")
  print("="*60)
#Перебирает всех сотрудников в онтологии
  for employee in onto.Специалист.instances():
          print(f"\nСотрудник: {employee.имя_сотрудника[0]}")

          #Печатает все знания сотрудника, которые были загружены из employees.csv (свойство обладает_знанием).
          print("Знания:")
          for knowledge in employee.обладает_знанием:
              print(f"  - {knowledge.название[0]}")
    
          print("Компетенции:")
          for competency in employee.обладает_компетенцией:
              print(f"  - {competency.название[0]}")

          print("Может занимать должности:")
          for position in employee.может_занимать_должность:
              print(f"  - {position.name}")

          print("-" * 30)

print_position_for_employee()


АНАЛИЗ ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ У СОТРУДНИКОВ

Сотрудник: Иванов Алексей Сергеевич
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Компетенции:
  - Программирование и алгоритмы
  - Машинное обучение
  - Развертывание ML-моделей
Может занимать должности:
  - ML Engineer
  - Computer Vision Engineer
------------------------------

Сотрудник: Петрова Мария Владимировна
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Компетенции:
  - Программирование и алгоритмы
  - Машинное обучение
  - Развертывание ML-моделей
Может занимать должности:
  - ML Engineer
  - Computer Vision Engineer
------------------------------

Сотрудник: Сидоров Дмитрий Петрович
Знания:
  - Математическая статистика
  - Линейная алгебра
  - SQL и базы данных
Компетенции:
  - Обработка и анализ данных
Может занимать должности:
  - Data Analyst
------------------------------

Сотрудн

In [34]:
# Убедимся, что у сотрудников достаточно знаний для получения компетенций
def check_employee_competencies(employee_instances, competency_instances):
    """Проверяем, какие компетенции могут получить сотрудники"""
    print("\n" + "="*60)
    print("ПРОВЕРКА ВОЗМОЖНЫХ КОМПЕТЕНЦИЙ СОТРУДНИКОВ")
    print("="*60)

    for employee_id, employee in employee_instances.items():
        employee_name = employee.имя_сотрудника[0]
        print(f"\n{employee_name}:")

        for competency_id, competency in competency_instances.items():
            comp_name = competency.название[0]
            required_knowledge = set(competency.требует_знание)
            employee_knowledge = set(employee.обладает_знанием)

            # Проверяем, есть ли у сотрудника все необходимые знания
            if required_knowledge.issubset(employee_knowledge):
                print(f"  Может получить компетенцию: {comp_name}")
                # Присваиваем компетенцию
                if competency not in employee.обладает_компетенцией:
                    employee.обладает_компетенцией.append(competency)
            else:
                missing_knowledge = required_knowledge - employee_knowledge
                if missing_knowledge:
                    print(f"  Не хватает для '{comp_name}':")
                    for know in missing_knowledge:
                        print(f"      - {know.название[0]}")


In [35]:
check_employee_competencies(employees, competencies)


ПРОВЕРКА ВОЗМОЖНЫХ КОМПЕТЕНЦИЙ СОТРУДНИКОВ

Иванов Алексей Сергеевич:
  Не хватает для 'Математическая подготовка':
      - Теория вероятностей
      - Линейная алгебра
  Может получить компетенцию: Программирование и алгоритмы
  Может получить компетенцию: Машинное обучение
  Не хватает для 'Глубокое обучение':
      - Обработка естественного языка (NLP)
      - Компьютерное зрение
  Не хватает для 'Обработка и анализ данных':
      - SQL и базы данных
  Не хватает для 'Работа с большими данными':
      - SQL и базы данных
      - Работа с большими данными (Spark, Hadoop)
  Не хватает для 'Визуализация и представление данных':
      - Визуализация данных
  Может получить компетенцию: Развертывание ML-моделей

Петрова Мария Владимировна:
  Не хватает для 'Математическая подготовка':
      - Теория вероятностей
      - Линейная алгебра
  Может получить компетенцию: Программирование и алгоритмы
  Может получить компетенцию: Машинное обучение
  Не хватает для 'Глубокое обучение':
      -

In [36]:
#Эта функция анализирует, каких знаний не хватает каждому сотруднику для получения той или иной компетенции, 
#и автоматически присваивает компетенции, если все требуемые знания присутствуют.
def check_employee_positions(employee_instances, position_instances):
    """Проверяем, какие должности могут занять сотрудники"""
    print("\n" + "="*60)
    print("ПРОВЕРКА ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ СОТРУДНИКОВ")
    print("="*60)
    #Принимает словари: employee_instances (сотрудники) и competency_instances (компетенции).
    for employee_id, employee in employee_instances.items():
        #Перебирает всех сотрудников, выводит имя.
        employee_name = employee.имя_сотрудника[0]
        print(f"\n{employee_name}:")

            # Для каждой компетенции превращает списки знаний (объекты) в множества (set). 
        #Множества удобны для сравнения: быстро проверять, входит ли одно во второе.
        for position_id, position in position_instances.items():
            pos_name = position.name
            required_competencies = set(position.требует_компетенцию)
            employee_competencies = set(employee.обладает_компетенцией)

            # Проверяем, есть ли у сотрудника все необходимые компетенции
            #Если все требуемые знания есть у сотрудника (проверка через issubset)
            if required_competencies.issubset(employee_competencies):
                print(f"   Может занять должность: {pos_name}")
                # Присваиваем должность
                if position not in employee.может_занимать_должность:
                    employee.может_занимать_должность.append(position)
            else:
                missing_competencies = required_competencies - employee_competencies
                if missing_competencies:
                    #Если знаний не хватает, вычисляется разность множеств (чего не хватает), и выводится список недостающих знаний.
                    print(f"   Не хватает для '{pos_name}':")
                    for comp in missing_competencies:
                        print(f"      - {comp.название[0]}")

# Применяем проверки

check_employee_positions(employees, positions)



ПРОВЕРКА ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ СОТРУДНИКОВ

Иванов Алексей Сергеевич:
   Может занять должность: ML Engineer
   Не хватает для 'Data Scientist':
      - Математическая подготовка
   Не хватает для 'Data Analyst':
      - Обработка и анализ данных
   Не хватает для 'NLP Engineer':
      - Глубокое обучение
   Может занять должность: Computer Vision Engineer

Петрова Мария Владимировна:
   Может занять должность: ML Engineer
   Не хватает для 'Data Scientist':
      - Математическая подготовка
   Не хватает для 'Data Analyst':
      - Обработка и анализ данных
   Не хватает для 'NLP Engineer':
      - Глубокое обучение
   Может занять должность: Computer Vision Engineer

Сидоров Дмитрий Петрович:
   Не хватает для 'ML Engineer':
      - Машинное обучение
      - Программирование и алгоритмы
   Не хватает для 'Data Scientist':
      - Машинное обучение
      - Программирование и алгоритмы
      - Математическая подготовка
   Может занять должность: Data Analyst
   Не хватает для 'NLP Engi

# Задание 1:  Добавление необходимой компетенции в должность

Добавить требование, что компетениция NLP Enginner содержит знание по глубокому обучению.

<B>Важно, что добавление в онтологию новых данных в Питоне должно сопровождаться созданием новой онтологии.</B>


In [37]:
positions

{1: ml_specialist_v1.ML Engineer,
 2: ml_specialist_v1.Data Scientist,
 3: ml_specialist_v1.Data Analyst,
 4: ml_specialist_v1.NLP Engineer,
 5: ml_specialist_v1.Computer Vision Engineer}

In [38]:
positions[4].требует_компетенцию

[ml_specialist_v1.компетенция_2, ml_specialist_v1.компетенция_4]

In [39]:
competencies[2].название

['Программирование и алгоритмы']

In [40]:
for i, item in competencies.items():
  print (i, ' ', item.название)


1   ['Математическая подготовка']
2   ['Программирование и алгоритмы']
3   ['Машинное обучение']
4   ['Глубокое обучение']
5   ['Обработка и анализ данных']
6   ['Работа с большими данными']
7   ['Визуализация и представление данных']
8   ['Развертывание ML-моделей']


In [41]:
positions[4].требует_компетенцию.append(competencies[4])

###

## Задание 2:
Добавьте требование в онтологию: должность Data Scientist требует компетенции "Математическая подготовка"

## Задание 3:
Добавьте сотруднику Федорову Максиму Андреевичу необходимые знания, чтобы он мог претендовать на должность Computer Vision Engineer

## Задание 4:
Добавьте требование в онтологию нового сотрудника, укажите в качестве имя_сотрудника ваше ФИО. Добавьте знания для данного сотрудника, чтобы он смог претендовать на все должности текущей онтологии